In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 현재 작동 중인 디바이스: {device} (cuda로 떠야 정상입니다!)")

In [ ]:
file_path = "cleaned_nhis_data_nan.csv"
df = pd.read_csv(file_path)

impute_vars = [
    'AGEP_A', 'SLPHOURS_A', 'DRKAVG12M_A', 
    'MODFREQW_A', 'VIGFREQW_A', 'MODNR_A', 'VIGNR_A',
    'PHSTAT_A', 'FDSCAT3_A'
]
impute_vars = [var for var in impute_vars if var in df.columns]
df_to_impute = df[impute_vars].copy()

mask = (~df_to_impute.isna()).astype(float).values
df_temp_filled = df_to_impute.fillna(0)

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df_temp_filled)

X_tensor = torch.FloatTensor(scaled_data)
M_tensor = torch.FloatTensor(mask)

dataset = TensorDataset(X_tensor, M_tensor)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

In [ ]:
dim = len(impute_vars)
h_dim = 32

class Generator(nn.Module):
    def __init__(self, dim, h_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, dim),
            nn.Sigmoid()
        )
    def forward(self, x, m):
        inputs = torch.cat([x, m], dim=1)
        return self.net(inputs)

class Discriminator(nn.Module):
    def __init__(self, dim, h_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, h_dim),
            nn.ReLU(),
            nn.Linear(h_dim, dim),
            nn.Sigmoid()
        )
    def forward(self, x, h):
        inputs = torch.cat([x, h], dim=1)
        return self.net(inputs)


G = Generator(dim, h_dim).to(device)
D = Discriminator(dim, h_dim).to(device)

optimizer_G = optim.Adam(G.parameters(), lr=0.001)
optimizer_D = optim.Adam(D.parameters(), lr=0.001)

In [ ]:
epochs = 500
hint_rate = 0.9
alpha = 10

print("🚀 GAIN (CUDA 가속) 학습 시작...")

for epoch in range(epochs):
    for X_batch, M_batch in dataloader:
        batch_size = X_batch.size(0)
        

        X_batch = X_batch.to(device)
        M_batch = M_batch.to(device)
        Z = torch.rand(batch_size, dim).to(device)
        
        X_hat = X_batch * M_batch + Z * (1 - M_batch)
        

        H_m = torch.rand(batch_size, dim).to(device) < hint_rate
        H_m = H_m.float()
        Hint = M_batch * H_m + 0.5 * (1 - H_m)
        

        optimizer_D.zero_grad()
        G_sample = G(X_hat, M_batch)
        X_tilde = X_batch * M_batch + G_sample * (1 - M_batch) 
        D_prob = D(X_tilde, Hint)
        
        D_loss = -torch.mean(M_batch * torch.log(D_prob + 1e-8) + \
                             (1 - M_batch) * torch.log(1. - D_prob + 1e-8))
        D_loss.backward(retain_graph=True)
        optimizer_D.step()
        

        optimizer_G.zero_grad()
        G_sample = G(X_hat, M_batch)
        X_tilde = X_batch * M_batch + G_sample * (1 - M_batch)
        D_prob = D(X_tilde, Hint)
        
        G_loss_BCE = -torch.mean((1 - M_batch) * torch.log(D_prob + 1e-8))
        G_loss_MSE = torch.mean((M_batch * X_batch - M_batch * G_sample)**2) / torch.mean(M_batch)
        
        G_loss = G_loss_BCE + alpha * G_loss_MSE
        G_loss.backward()
        optimizer_G.step()
        
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] | D_Loss: {D_loss.item():.4f} | G_Loss: {G_loss.item():.4f}')

In [ ]:
G.eval()
with torch.no_grad():

    X_tensor = X_tensor.to(device)
    M_tensor = M_tensor.to(device)
    Z_full = torch.rand(len(X_tensor), dim).to(device)
    
    X_hat_full = X_tensor * M_tensor + Z_full * (1 - M_tensor)
    
    G_sample_full = G(X_hat_full, M_tensor)
    imputed_scaled = X_tensor * M_tensor + G_sample_full * (1 - M_tensor)


imputed_data = scaler.inverse_transform(imputed_scaled.cpu().numpy())
df_imputed = pd.DataFrame(imputed_data, columns=impute_vars)

df_imputed = df_imputed.round(0)

In [ ]:
df_final = df.copy()
df_final[impute_vars] = df_imputed

output_file = "final_nhis_data_gain_imputed_cuda.csv"
df_final.to_csv(output_file, index=False)

print(f"\n✅ GPU 파워로 순식간에 보간 완료! 데이터가 '{output_file}'로 저장되었습니다.")